# MODELLING PHASE:
## Project Overview
This notebook documents the **Modelling Phase** of the **Alternative Credit Scoring** project. The goal is to build a predictive model that identifies financial risk profiles using non-traditional behavioral and demographic data to enhance financial inclusion.

## Data Source
The analysis utilizes the **2021 FinAccess Household Survey (KNBS)**, featuring:

- **Demographics:** Age, gender, and education  
- **Economic Indicators:** Income groups and asset ownership  
- **Behavioral Data:** Usage of mobile money, banking, and informal groups (Chamas)  

## Technical Workflow
- **Preprocessing:** Feature scaling and addressing class imbalances  
- **Model Selection:** Comparative analysis of Logistic Regression, Random Forest, and XGBoost  
- **Evaluation:** Performance is measured via ROC-AUC scores to determine classification accuracy  
- **Feature Importance:** Identifying the primary demographic and behavioral drivers of financial risk  

## Objective
To develop a validated scoring engine that assists in **data-driven lending decisions** for populations lacking traditional credit histories.

In [ ]:
#we import the libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
# we load the cleaned and encoded dataset
df=pd.read_csv('C:\\Users\\Win\\Alternative credit scoring\\Data\\Modelling_data.csv')
df.head()

In [ ]:
df.columns

In [ ]:
df.shape

In [ ]:
# we import our models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score


In [ ]:
# 1. Preparing X and y
X = df.drop('is_high_risk_target', axis=1)
y = df['is_high_risk_target']

# 2. Split the data (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


In [ ]:
# 3. Scaling (Essential for Logistic Regression weights)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
X_train.shape

In [ ]:
#checking for Nan values
#(we were doing this cause at some point our kernel died, and we were investigating why, we reactivated it at the anaconda prompt,and all was good)
np.isnan(X_train_scaled).sum()

In [ ]:
#checking for infinite values
np.isinf(X_train_scaled).sum()


In [ ]:
# 4. Model 1: Logistic Regression (The Baseline)
lr_model = LogisticRegression(class_weight='balanced', solver='liblinear',max_iter=1000,random_state=42)
lr_model.fit(X_train_scaled, y_train)
lr_probs = lr_model.predict_proba(X_test_scaled)[:, 1]


In [ ]:
# 5. Model 2: Random Forest (The Heavy Lifter)
rf_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf_model.fit(X_train, y_train) # RF doesn't strictly need scaling
rf_probs = rf_model.predict_proba(X_test)[:, 1]


In [ ]:
# 6. Print Performance
print(f"Logistic Regression ROC-AUC: {roc_auc_score(y_test, lr_probs):.4f}")
print(f"Random Forest ROC-AUC:       {roc_auc_score(y_test, rf_probs):.4f}")

In [ ]:
from xgboost import XGBClassifier

# 1. we initialize XGBoost
# We use scale_pos_weight to handle the fact that High Risk is only ~35% of the data
# This is mathematically similar to 'class_weight=balanced' in the other models
xgb_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    scale_pos_weight=(len(y_train) - sum(y_train)) / sum(y_train), 
    random_state=42,
    eval_metric='logloss'
)

In [ ]:
# 2.we train XGBoost
xgb_model.fit(X_train, y_train)
xgb_probs = xgb_model.predict_proba(X_test)[:, 1]


In [ ]:
# 3. Final Performance Comparison
print(f"Logistic Regression ROC-AUC: {roc_auc_score(y_test, lr_probs):.4f}")
print(f"Random Forest ROC-AUC:       {roc_auc_score(y_test, rf_probs):.4f}")
print(f"XGBoost ROC-AUC:             {roc_auc_score(y_test, xgb_probs):.4f}")

### FEATURE IMPORTANCE

In [ ]:
# 1. we pull Importance from Logistic Regression (Absolute Coefficients)
lr_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': np.abs(lr_model.coef_[0])
}).sort_values(by='Importance', ascending=False)


In [ ]:
# 2. we pull Importance from Random Forest
rf_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values(by='Importance', ascending=False)


In [ ]:
# 3. we pull Importance from XGBoost
xgb_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': xgb_model.feature_importances_
}).sort_values(by='Importance', ascending=False)


In [ ]:
# 4.we now Plot the feature importances 
plt.figure(figsize=(20, 10))

plt.subplot(1, 3, 1)
sns.barplot(x='Importance', y='Feature',hue='Importance', data=lr_importance.head(10), palette='Reds_r')
plt.title('Top 10: Logistic Regression (Weights)')

plt.subplot(1, 3, 2)
sns.barplot(x='Importance', y='Feature', hue='Importance',data=rf_importance.head(10), palette='Blues_r')
plt.title('Top 10: Random Forest (Gini)')

plt.subplot(1, 3, 3)
sns.barplot(x='Importance', y='Feature',hue='Importance', data=xgb_importance.head(10), palette='Greens_r')
plt.title('Top 10: XGBoost (Gain)')

plt.tight_layout()
plt.show()

## HYPERPARMETER TUNING FOR XGBOOST:

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

# 1. Define the Parameter Space
param_dist = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 4, 5, 6],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'subsample': [0.6, 0.8, 1.0],
    'gamma': [0, 0.1, 0.2]
}


In [ ]:
# 2. we initialize Randomized Search
# We do 5-fold cross-validation and 20 "iterations" (tries)
random_search = RandomizedSearchCV(
    xgb_model, 
    param_distributions=param_dist, 
    n_iter=20, 
    scoring='roc_auc', 
    cv=5, 
    verbose=1, 
    n_jobs=-1, 
    random_state=42
)

In [ ]:
# 3. we fit the search
random_search.fit(X_train, y_train)

In [ ]:
# 4. we get the best score and model
print(f"Best Parameters: {random_search.best_params_}")
print(f"Best Cross-Val ROC-AUC: {random_search.best_score_:.4f}")

In [ ]:
# 5. we evaluate on Test Set
best_xgb = random_search.best_estimator_
tuned_probs = best_xgb.predict_proba(X_test)[:, 1]
print(f"Tuned XGBoost Test ROC-AUC: {roc_auc_score(y_test, tuned_probs):.4f}")

### METRICS:

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

In [ ]:
# 1. Get hard predictions (0 or 1)
y_pred = best_xgb.predict(X_test)

In [ ]:
# 2. Generate Confusion Matrix
cm = confusion_matrix(y_test, y_pred)


In [ ]:
# 3. Plot
plt.figure(figsize=(8, 6))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Low Risk', 'High Risk']).plot(cmap='Blues')
plt.title('Confusion Matrix: Tuned XGBoost')
plt.show()

In [ ]:
# 4. Detailed Report
print("--- Classification Report ---")
print(classification_report(y_test, y_pred))

#### MODELS SAVING:

In [154]:
# we save our models

import joblib

# Save Logistic Regression
joblib.dump(lr_model, 'logistic_regression_model.joblib')

# Save Random Forest
joblib.dump(rf_model, 'random_forest_model.joblib')

# Save XGBoost (using its native format is recommended)
joblib.dump(xgb_model, 'xgb_credit_model.joblib')

# Important: Also save your scaler! 
# You will need it to scale new data before prediction.
joblib.dump(scaler, 'scaler.joblib')

print("Models and scaler saved successfully!")

Models and scaler saved successfully!


##  Final Reflections: Feature Engineering Potential
While our tuned XGBoost model achieved a solid baseline, further performance gains ($>0.70$ AUC) would likely require **Feature Engineering** rather than further hyperparameter tuning.

- **Interaction Terms:** Creating features that combine demographic data (Age) with behavioral data (Mobile Banking) would capture the "hidden" signals in household economics.
- **Normalization:** Developing a 'Wealth Index' by combining income with asset ownership or education would provide a more stable predictor than volatile monthly income alone.
- **Conclusion:** The plateau in performance ($0.6681$) suggests that we have extracted the maximum signal possible from the raw features; the next stage of model maturity involves synthesizing new behavioral indicators.